## 1. Setup and Imports

In [ ]:
import os
import time
import math
from typing import Optional

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torchaudio
import torchaudio.functional as F
from torchaudio.datasets import SPEECHCOMMANDS
from torch.utils.data import DataLoader, Subset
from torch.optim import Adam
from tqdm import tqdm

# Для подсчёта FLOPs (пример: thop или fvcore)
# pip install thop (если не установлен)
try:
    from thop import profile, clever_format
except ImportError:
    print("thop not installed. Install with: pip install thop")

# Проверка устройства
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Фиксация seed для воспроизводимости
def set_seed(seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed()

## 2. Implement LogMelFilterBanks Layer

In [ ]:
import torch
import torch.nn as nn
import torchaudio.functional as ta_F

class LogMelFilterBanks(nn.Module):
    def __init__(
            self,
            n_fft: int = 400,
            samplerate: int = 16000,
            hop_length: int = 160,
            n_mels: int = 80,
            pad_mode: str = 'reflect',
            power: float = 2.0,
            normalize_stft: bool = False,
            onesided: bool = True,
            center: bool = True,
            return_complex: bool = True,
            f_min_hz: float = 0.0,
            f_max_hz: Optional[float] = None,
            norm_mel: Optional[str] = None,
            mel_scale: str = 'htk'
        ):
        super(LogMelFilterBanks, self).__init__()
        self.n_fft = n_fft
        self.samplerate = samplerate
        self.window_length = n_fft
        self.window = torch.hann_window(self.window_length)

        self.hop_length = hop_length
        self.center = center
        self.return_complex = return_complex
        self.onesided = onesided
        self.normalize_stft = normalize_stft
        self.pad_mode = pad_mode
        self.power = power

        self.n_mels = n_mels
        self.f_min_hz = f_min_hz
        self.f_max_hz = f_max_hz if f_max_hz is not None else samplerate / 2.0
        self.norm_mel = norm_mel
        self.mel_scale = mel_scale

        self.n_freqs = n_fft // 2 + 1 if onesided else n_fft
        self.register_buffer('mel_fbanks', self._init_melscale_fbanks())

    def _init_melscale_fbanks(self):
        fbanks = ta_F.melscale_fbanks(
            n_freqs=self.n_freqs,
            f_min=self.f_min_hz,
            f_max=self.f_max_hz,
            n_mels=self.n_mels,
            sample_rate=self.samplerate,
            norm=self.norm_mel,
            mel_scale=self.mel_scale,
        )
        # В некоторых версиях torchaudio фильтры возвращаются в форме (n_freqs, n_mels)
        if fbanks.shape[0] == self.n_freqs and fbanks.shape[1] == self.n_mels:
            fbanks = fbanks.T
        return fbanks

    def spectrogram(self, x):
        return torch.stft(
            x,
            n_fft=self.n_fft,
            hop_length=self.hop_length,
            win_length=self.window_length,
            window=self.window.to(x.device),
            center=self.center,
            pad_mode=self.pad_mode,
            normalized=self.normalize_stft,
            onesided=self.onesided,
            return_complex=self.return_complex
        )

    def forward(self, x):
        spec = self.spectrogram(x)
        if self.return_complex:
            power_spec = spec.abs() ** self.power
        else:
            power_spec = (spec[..., 0] ** 2 + spec[..., 1] ** 2) ** (self.power / 2)
        # Убедимся, что ось частот на месте (B, n_freqs, T)
        if power_spec.dim() == 3 and power_spec.shape[1] != self.n_freqs:
            # Если форма (B, T, n_freqs) -> меняем
            power_spec = power_spec.transpose(1, 2)
        mel_spec = torch.matmul(self.mel_fbanks, power_spec)
        log_mel = torch.log(mel_spec + 1e-6)
        return log_mel

## 3. Correctness check (vs torchaudio)

In [ ]:
def load_audio_sample(sr=16000):
    """Пытается загрузить реальный wav из SPEECHCOMMANDS (папка ./data), иначе генерирует синусоиду."""
    data_dir = "./data"
    sample_path = None
    if os.path.exists(data_dir):
        for root, dirs, files in os.walk(data_dir):
            for file in files:
                if file.endswith('.wav') and ('yes' in root.lower() or 'no' in root.lower()):
                    sample_path = os.path.join(root, file)
                    break
            if sample_path:
                break

    if sample_path and os.path.exists(sample_path):
        waveform, orig_sr = torchaudio.load(sample_path)
        if orig_sr != sr:
            resampler = torchaudio.transforms.Resample(orig_sr, sr)
            waveform = resampler(waveform)
        return waveform
    else:
        print("Реальный файл не найден, генерируем синусоиду 440 Гц")
        t = torch.linspace(0, 1.0, int(sr * 1.0))
        freq = 440.0
        waveform = 0.5 * torch.sin(2 * np.pi * freq * t)
        return waveform.unsqueeze(0)

# Загружаем сигнал
signal = load_audio_sample(16000)
sr = 16000
print(f"Сигнал shape: {signal.shape}, sr={sr}")

n_mels_test = 80
hop_len = 160
n_fft = 400
power = 2.0

our_layer = LogMelFilterBanks(
    n_fft=n_fft,
    samplerate=sr,
    hop_length=hop_len,
    n_mels=n_mels_test,
    power=power,
    center=True,
    onesided=True,
    return_complex=True
)

mel_torch = torchaudio.transforms.MelSpectrogram(
    sample_rate=sr,
    n_fft=n_fft,
    hop_length=hop_len,
    n_mels=n_mels_test,
    power=power,
    center=True,
    onesided=True,
    norm=None,
    mel_scale='htk'
)

with torch.no_grad():
    logmel_ours = our_layer(signal)
    spec_torch = mel_torch(signal)
    logmel_torch = torch.log(spec_torch + 1e-6)

assert logmel_ours.shape == logmel_torch.shape, f"Shape mismatch: {logmel_ours.shape} vs {logmel_torch.shape}"

max_diff = torch.max(torch.abs(logmel_ours - logmel_torch)).item()
mean_diff = torch.mean(torch.abs(logmel_ours - logmel_torch)).item()
print(f"Максимальная абсолютная разница: {max_diff:.6f}")
print(f"Средняя абсолютная разница: {mean_diff:.6f}")

atol = 1e-5
rtol = 1e-4
try:
    torch.testing.assert_allclose(logmel_ours, logmel_torch, atol=atol, rtol=rtol)
    print(f"✅ Реализация корректна (совпадает с эталоном в пределах atol={atol}, rtol={rtol})")
except AssertionError as e:
    print(f"⚠️ Реализация не совпадает с эталоном: {e}")

plt.figure(figsize=(14, 6))
plt.subplot(1, 3, 1)
plt.imshow(logmel_ours[0].cpu().numpy(), aspect='auto', origin='lower')
plt.title('Our LogMelFilterBanks')
plt.colorbar()
plt.subplot(1, 3, 2)
plt.imshow(logmel_torch[0].cpu().numpy(), aspect='auto', origin='lower')
plt.title('torchaudio MelSpectrogram + log')
plt.colorbar()
plt.subplot(1, 3, 3)
diff_img = (logmel_ours - logmel_torch)[0].cpu().numpy()
im = plt.imshow(diff_img, aspect='auto', origin='lower', cmap='RdBu')
plt.title('Difference (ours - torch)')
plt.colorbar(im)
plt.tight_layout()
plt.savefig('logmel_comparison.png', dpi=150)
plt.show()

## 4. Подготовка датасета Speech Commands (бинарная классификация "yes"/"no")

In [ ]:

import os
import torch
import torchaudio
from torchaudio.datasets import SPEECHCOMMANDS
from torch.utils.data import Subset, DataLoader
import numpy as np
import glob

data_dir = "./data"
os.makedirs(data_dir, exist_ok=True)

def get_speechcommands_subsets(root, subset_ratio=1.0):
    full_dataset = SPEECHCOMMANDS(root=root, download=True)

    # Поиск validation_list.txt и testing_list.txt
    val_file = None
    test_file = None
    for path in glob.glob(os.path.join(root, '**', 'validation_list.txt'), recursive=True):
        val_file = path
        test_file = os.path.join(os.path.dirname(path), 'testing_list.txt')
        if os.path.exists(test_file):
            break
    if val_file is None:
        raise FileNotFoundError("Не найдены validation_list.txt и testing_list.txt")

    with open(val_file) as f:
        val_set_paths = set(line.strip() for line in f)
    with open(test_file) as f:
        test_set_paths = set(line.strip() for line in f)

    # Определяем префикс для удаления из _walker
    if hasattr(full_dataset, '_walker') and len(full_dataset._walker) > 0:
        first = full_dataset._walker[0]
        idx = first.find("speech_commands_v0.02/")
        if idx != -1:
            prefix_to_strip = first[:idx + len("speech_commands_v0.02/")]
        else:
            prefix_to_strip = ''
    else:
        prefix_to_strip = ''

    print(f"Префикс для удаления: {prefix_to_strip}")

    train_idx, val_idx, test_idx = [], [], []
    train_lbl, val_lbl, test_lbl = [], [], []

    for i, (wav, sr, lbl, spk, utt) in enumerate(full_dataset):
        lbl_low = lbl.lower()
        if lbl_low not in ('yes', 'no'):
            continue
        bin_label = 1 if lbl_low == 'yes' else 0

        walker_path = full_dataset._walker[i] if hasattr(full_dataset, '_walker') else f"{lbl}/{spk}_{utt}.wav"
        if prefix_to_strip:
            walker_path = walker_path.replace(prefix_to_strip, '', 1)

        if walker_path in val_set_paths:
            val_idx.append(i)
            val_lbl.append(bin_label)
        elif walker_path in test_set_paths:
            test_idx.append(i)
            test_lbl.append(bin_label)
        else:
            train_idx.append(i)
            train_lbl.append(bin_label)

    print(f"Найдено в тренировке: {len(train_idx)} (yes: {sum(train_lbl)}), валидация: {len(val_idx)} (yes: {sum(val_lbl)}), тест: {len(test_idx)} (yes: {sum(test_lbl)})")

    if subset_ratio < 1.0:
        train_idx = train_idx[:int(len(train_idx)*subset_ratio)]
        val_idx = val_idx[:int(len(val_idx)*subset_ratio)]
        test_idx = test_idx[:int(len(test_idx)*subset_ratio)]
        train_lbl = train_lbl[:len(train_idx)]
        val_lbl = val_lbl[:len(val_idx)]
        test_lbl = test_lbl[:len(test_idx)]

    train_set = Subset(full_dataset, train_idx)
    val_set = Subset(full_dataset, val_idx)
    test_set = Subset(full_dataset, test_idx)

    return (train_set, val_set, test_set,
            torch.tensor(train_lbl, dtype=torch.long),
            torch.tensor(val_lbl, dtype=torch.long),
            torch.tensor(test_lbl, dtype=torch.long))

subset_ratio = 1.0   # можно увеличить до 1.0 при финальном обучении
train_set, val_set, test_set, train_labels, val_labels, test_labels = get_speechcommands_subsets(
    data_dir, subset_ratio=subset_ratio
)

print(f"Train samples: {len(train_set)}, Val: {len(val_set)}, Test: {len(test_set)}")
print(f"Train class distribution: no = {(train_labels == 0).sum().item()}, yes = {(train_labels == 1).sum().item()}")
print(f"Val class distribution: no = {(val_labels == 0).sum().item()}, yes = {(val_labels == 1).sum().item()}")
print(f"Test class distribution: no = {(test_labels == 0).sum().item()}, yes = {(test_labels == 1).sum().item()}")

# %%
# Преобразование аудио в Mel-спектрограмму (только признаки)
class AudioToMelTransform:
    def __init__(self, mel_layer):
        self.mel_layer = mel_layer
    def __call__(self, waveform, sample_rate, label, speaker_id, utterance_number):
        # Приводим к (1, T)
        if waveform.dim() == 2 and waveform.size(0) == 1:
            waveform = waveform.squeeze(0)
        elif waveform.dim() == 2 and waveform.size(0) > 1:
            waveform = waveform[0]
        waveform = waveform.unsqueeze(0)
        mel_spec = self.mel_layer(waveform)   # (1, n_mels, T)
        return mel_spec.squeeze(0)            # (n_mels, T)

# Инициализируем слой Mel (n_mels пока 80)
initial_n_mels = 80
mel_layer = LogMelFilterBanks(samplerate=16000, n_mels=initial_n_mels)
transform = AudioToMelTransform(mel_layer)

# Датасет с числовыми метками
class MelDataset:
    def __init__(self, subset, transform, labels):
        self.subset = subset
        self.transform = transform
        self.labels = labels
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        item = self.subset[idx]
        waveform, sr, _, speaker_id, utterance_number = item
        mel_spec = self.transform(waveform, sr, None, speaker_id, utterance_number)
        return mel_spec, self.labels[idx]

train_mel = MelDataset(train_set, transform, train_labels)
val_mel = MelDataset(val_set, transform, val_labels)
test_mel = MelDataset(test_set, transform, test_labels)

def collate_fn(batch):
    mels, labels = zip(*batch)
    max_len = max(m.shape[1] for m in mels)
    padded_mels = []
    for m in mels:
        pad_len = max_len - m.shape[1]
        if pad_len > 0:
            m = torch.nn.functional.pad(m, (0, pad_len))
        padded_mels.append(m)
    return torch.stack(padded_mels), torch.tensor(labels, dtype=torch.long)

BATCH_SIZE = 32
train_loader = DataLoader(train_mel, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader = DataLoader(val_mel, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)
test_loader = DataLoader(test_mel, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)

# Быстрая проверка
for batch_mels, batch_labels in train_loader:
    print("Batch mels shape:", batch_mels.shape)
    print("Batch labels:", batch_labels)
    break

## 5. Определение модели CNN (Conv1d) и вспомогательных функций


In [ ]:
!pip install thop

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from thop import profile, clever_format

class SimpleCNN(nn.Module):
    """
    Простая свёрточная сеть для классификации log-mel спектрограмм.
    Args:
        n_mels (int): количество мел-фильтров (входных каналов)
        n_classes (int): число классов (2 для yes/no)
        groups (int): параметр groups для Conv1d (групповые свёртки)
    """
    def __init__(self, n_mels=80, n_classes=2, groups=1):
        super(SimpleCNN, self).__init__()
        # Первый свёрточный слой: 32 выхода, ядро 3, padding=1
        self.conv1 = nn.Conv1d(in_channels=n_mels, out_channels=32,
                               kernel_size=3, stride=1, padding=1, groups=groups)
        self.bn1 = nn.BatchNorm1d(32)
        # Второй свёрточный слой: 64 выхода
        self.conv2 = nn.Conv1d(in_channels=32, out_channels=64,
                               kernel_size=3, stride=1, padding=1, groups=groups)
        self.bn2 = nn.BatchNorm1d(64)
        # Глобальный пулинг по временной оси
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        # Классификатор
        self.fc = nn.Linear(64, n_classes)
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        # x: (batch, n_mels, time)
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = self.global_pool(x)          # (batch, 64, 1)
        x = x.squeeze(-1)                # (batch, 64)
        x = self.dropout(x)
        x = self.fc(x)                   # (batch, n_classes)
        return x

def count_parameters(model):
    """Возвращает количество обучаемых параметров модели."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def compute_flops(model, input_shape=(1, 80, 100), device='cpu'):
    """
    Вычисляет количество FLOPs для модели.
    input_shape: (batch_size, n_mels, time_frames)
    """
    model = model.to(device)
    model.eval()
    input_tensor = torch.randn(input_shape).to(device)
    flops, params = profile(model, inputs=(input_tensor,), verbose=False)
    return flops, params

def train_epoch(model, loader, optimizer, criterion, device):
    """Одна эпоха обучения."""
    model.train()
    total_loss = 0.0
    total_samples = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        total_samples += x.size(0)
    return total_loss / total_samples

def validate(model, loader, criterion, device):
    """Валидация (потеря и accuracy)."""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits, y)
            total_loss += loss.item() * x.size(0)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += x.size(0)
    avg_loss = total_loss / total
    acc = correct / total
    return avg_loss, acc

# Проверка модели с текущими параметрами
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

model = SimpleCNN(n_mels=80, n_classes=2, groups=1).to(device)
param_count = count_parameters(model)
print(f"Number of parameters: {param_count}")

# Оценка FLOPs (примерный вход 80 mel-бинов, 100 временных фреймов)
flops, params = compute_flops(model, input_shape=(1, 80, 100), device=device)
flops_clean, params_clean = clever_format([flops, params], "%.3f")
print(f"FLOPs: {flops_clean}, Params: {params_clean}")

## 6. Эксперименты с количеством мел-фильтров (n_mels = 20, 40, 80)

In [ ]:
import time
import matplotlib.pyplot as plt

n_mels_list = [20, 40, 80]
results_mels = {}

EPOCHS = 10
LEARNING_RATE = 1e-3
BATCH_SIZE = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for n_mels in n_mels_list:
    print(f"\n{'='*50}\nTraining with n_mels = {n_mels}\n{'='*50}")

    mel_layer = LogMelFilterBanks(samplerate=16000, n_mels=n_mels)
    transform = AudioToMelTransform(mel_layer)

    train_mel = MelDataset(train_set, transform, train_labels)
    val_mel = MelDataset(val_set, transform, val_labels)
    test_mel = MelDataset(test_set, transform, test_labels)

    train_loader = DataLoader(train_mel, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2)
    val_loader = DataLoader(val_mel, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)
    test_loader = DataLoader(test_mel, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)

    model = SimpleCNN(n_mels=n_mels, n_classes=2, groups=1).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    train_losses, val_accs, epoch_times = [], [], []

    for epoch in range(1, EPOCHS+1):
        start = time.time()
        train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        epoch_times.append(time.time() - start)
        train_losses.append(train_loss)
        val_accs.append(val_acc)
        print(f"Epoch {epoch:2d}/{EPOCHS} | loss: {train_loss:.4f} | val_acc: {val_acc:.4f} | time: {epoch_times[-1]:.2f}s")

    test_loss, test_acc = validate(model, test_loader, criterion, device)
    params = count_parameters(model)
    print(f"Test accuracy: {test_acc:.4f}, Parameters: {params}")

    results_mels[n_mels] = {
        'train_losses': train_losses,
        'val_accs': val_accs,
        'epoch_times': epoch_times,
        'test_acc': test_acc,
        'params': params
    }

In [ ]:
plt.style.use('seaborn-v0_8-darkgrid')  # красивый стиль
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# График train loss (левый)
ax1 = axes[0]
for n_mels, res in results_mels.items():
    ax1.plot(res['train_losses'], marker='o', linewidth=2, label=f'n_mels={n_mels}')
ax1.set_xlabel('Epoch', fontsize=12)
ax1.set_ylabel('Train Loss', fontsize=12)
ax1.set_title('Training Loss', fontsize=14)
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.7)

# График validation accuracy (средний)
ax2 = axes[1]
for n_mels, res in results_mels.items():
    ax2.plot(res['val_accs'], marker='s', linewidth=2, label=f'n_mels={n_mels}')
ax2.set_xlabel('Epoch', fontsize=12)
ax2.set_ylabel('Validation Accuracy', fontsize=12)
ax2.set_title('Validation Accuracy', fontsize=14)
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.7)

# Правый график: test accuracy (левая ось) и количество параметров (правая ось)
ax3 = axes[2]
n_vals = list(results_mels.keys())
test_acc_vals = [results_mels[n]['test_acc'] for n in n_vals]
params_vals = [results_mels[n]['params'] / 1000 for n in n_vals]  # в тыс.

# Столбцы для accuracy
bars1 = ax3.bar([x - 0.2 for x in range(len(n_vals))], test_acc_vals, width=0.4,
                color='steelblue', label='Test Accuracy', alpha=0.8)
ax3.set_xlabel('n_mels', fontsize=12)
ax3.set_ylabel('Test Accuracy', fontsize=12, color='steelblue')
ax3.tick_params(axis='y', labelcolor='steelblue')
ax3.set_ylim(0.5, 1.0)
ax3.set_title('Test Accuracy & Parameters', fontsize=14)
ax3.grid(True, linestyle='--', alpha=0.3, axis='y')

# Вторая ось для параметров
ax4 = ax3.twinx()
bars2 = ax4.bar([x + 0.2 for x in range(len(n_vals))], params_vals, width=0.4,
                color='coral', label='Params (k)', alpha=0.8)
ax4.set_ylabel('Number of Parameters (thousands)', fontsize=12, color='coral')
ax4.tick_params(axis='y', labelcolor='coral')

# Общая легенда для двух осей
lines1, labels1 = ax3.get_legend_handles_labels()
lines2, labels2 = ax4.get_legend_handles_labels()
ax3.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

# Значения над столбцами
for i, (acc, param) in enumerate(zip(test_acc_vals, params_vals)):
    ax3.text(i - 0.2, acc + 0.01, f'{acc:.3f}', ha='center', fontsize=9, color='steelblue')
    ax4.text(i + 0.2, param + 1, f'{param:.0f}k', ha='center', fontsize=9, color='coral')

plt.tight_layout()
plt.savefig('n_mels_experiment.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Эксперименты с параметром groups в Conv1d (baseline с n_mels=80)

In [ ]:
# Выберем лучшее значение n_mels из предыдущих экспериментов (например, 80, либо то, что дало лучший баланс).
# Если не определились, возьмём n_mels=80 как наиболее информативный.

best_n_mels = 80   # можно подставить 40, если он показал лучшую точность
groups_list = [1, 2, 4, 8, 16]
results_groups = {}

EPOCHS = 10
LEARNING_RATE = 1e-3
BATCH_SIZE = 32
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Using device: {device}")
print(f"Fixed n_mels = {best_n_mels}")

# Создаём слой Mel для выбранного n_mels (один раз, чтобы не пересоздавать датасеты)
mel_layer_fixed = LogMelFilterBanks(samplerate=16000, n_mels=best_n_mels)
transform_fixed = AudioToMelTransform(mel_layer_fixed)

# Создаём датасеты один раз (они не зависят от groups)
train_mel_fixed = MelDataset(train_set, transform_fixed, train_labels)
val_mel_fixed = MelDataset(val_set, transform_fixed, val_labels)
test_mel_fixed = MelDataset(test_set, transform_fixed, test_labels)

train_loader_fixed = DataLoader(train_mel_fixed, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=2)
val_loader_fixed = DataLoader(val_mel_fixed, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)
test_loader_fixed = DataLoader(test_mel_fixed, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, num_workers=2)

for groups in groups_list:
    print(f"\n{'='*50}\nTraining with groups = {groups}\n{'='*50}")

    model = SimpleCNN(n_mels=best_n_mels, n_classes=2, groups=groups).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    criterion = nn.CrossEntropyLoss()

    train_losses = []
    val_accs = []
    epoch_times = []

    for epoch in range(1, EPOCHS+1):
        start_time = time.time()
        train_loss = train_epoch(model, train_loader_fixed, optimizer, criterion, device)
        val_loss, val_acc = validate(model, val_loader_fixed, criterion, device)
        epoch_time = time.time() - start_time

        train_losses.append(train_loss)
        val_accs.append(val_acc)
        epoch_times.append(epoch_time)
        print(f"Epoch {epoch:2d}/{EPOCHS} | train_loss: {train_loss:.4f} | val_acc: {val_acc:.4f} | time: {epoch_time:.2f}s")

    test_loss, test_acc = validate(model, test_loader_fixed, criterion, device)
    params = count_parameters(model)

    # Оценка FLOPs (входной размер: (1, best_n_mels, 100))
    flops, _ = compute_flops(model, input_shape=(1, best_n_mels, 100), device=device)

    results_groups[groups] = {
        'train_losses': train_losses,
        'val_accs': val_accs,
        'epoch_times': epoch_times,
        'test_acc': test_acc,
        'params': params,
        'flops': flops
    }
    print(f"Test accuracy: {test_acc:.4f}, Parameters: {params}, FLOPs: {flops:.2e}")

In [ ]:
groups_sorted = sorted(groups_list)
avg_epoch_time = [np.mean(results_groups[g]['epoch_times']) for g in groups_sorted]
params_vals = [results_groups[g]['params'] for g in groups_sorted]
flops_vals = [results_groups[g]['flops'] for g in groups_sorted]
test_acc_vals = [results_groups[g]['test_acc'] for g in groups_sorted]

plt.style.use('seaborn-v0_8-darkgrid')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# ---- График 1: Accuracy (левая ось) и Training time (правая) ----
ax1.plot(groups_sorted, test_acc_vals, marker='o', linewidth=2.5, markersize=8,
         color='#1f77b4', label='Test Accuracy')
ax1.set_xlabel('groups', fontsize=12)
ax1.set_ylabel('Test Accuracy', fontsize=12, color='#1f77b4')
ax1.tick_params(axis='y', labelcolor='#1f77b4')
ax1.set_ylim(0.7, 1.0)
ax1.grid(True, linestyle='--', alpha=0.5)

# Добавляем числовые подписи accuracy над точками
for i, (g, acc) in enumerate(zip(groups_sorted, test_acc_vals)):
    ax1.annotate(f'{acc:.3f}', (g, acc), textcoords="offset points", xytext=(0, 10),
                 ha='center', fontsize=9, color='#1f77b4')

# Вторая ось для времени
ax1b = ax1.twinx()
ax1b.plot(groups_sorted, avg_epoch_time, marker='s', linewidth=2.5, markersize=8,
          color='#ff7f0e', label='Avg Epoch Time (s)')
ax1b.set_ylabel('Avg Epoch Time (s)', fontsize=12, color='#ff7f0e')
ax1b.tick_params(axis='y', labelcolor='#ff7f0e')

# Объединяем легенду
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax1b.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='best', frameon=True)

ax1.set_title('Test Accuracy vs Training Time', fontsize=14)

# ---- График 2: Parameters (левая ось) и FLOPs (правая) ----
ax2.plot(groups_sorted, params_vals, marker='^', linewidth=2.5, markersize=8,
         color='#2ca02c', label='Params')
ax2.set_xlabel('groups', fontsize=12)
ax2.set_ylabel('Number of Parameters', fontsize=12, color='#2ca02c')
ax2.tick_params(axis='y', labelcolor='#2ca02c')
ax2.grid(True, linestyle='--', alpha=0.5)

# Подписи параметров
for i, (g, p) in enumerate(zip(groups_sorted, params_vals)):
    ax2.annotate(f'{p}', (g, p), textcoords="offset points", xytext=(0, 10),
                 ha='center', fontsize=9, color='#2ca02c')

# Вторая ось для FLOPs (логарифмическая)
ax2b = ax2.twinx()
ax2b.plot(groups_sorted, flops_vals, marker='D', linewidth=2.5, markersize=8,
          color='#d62728', label='FLOPs')
ax2b.set_ylabel('FLOPs (log scale)', fontsize=12, color='#d62728')
ax2b.tick_params(axis='y', labelcolor='#d62728')
ax2b.set_yscale('log')

# Объединяем легенду
lines3, labels3 = ax2.get_legend_handles_labels()
lines4, labels4 = ax2b.get_legend_handles_labels()
ax2.legend(lines3 + lines4, labels3 + labels4, loc='best', frameon=True)

ax2.set_title('Model Size (Params) vs Computational Cost (FLOPs)', fontsize=14)

plt.tight_layout()
plt.savefig('groups_experiment_compact.png', dpi=150, bbox_inches='tight')
plt.show()